In [2]:
!pip install tqdm


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [4]:
"""
Enhanced NASA POWER Data Extraction with Dense Synthetic Grid
Creates 100+ locations through grid densification and spatial interpolation
Time range: 2018-01-01 to 2024-12-31
Spatial extent: Washington State (45.53-49°N, -124.77 to -116.92°W)
"""

import os
import fsspec
import pandas as pd
import xarray as xr
import numpy as np
from datetime import datetime
from scipy.interpolate import griddata
from tqdm.auto import tqdm

def create_dense_grid(lat_range, lon_range, spacing=0.25):
    """
    Create dense synthetic grid by subdividing the region
    
    Parameters:
    -----------
    lat_range : tuple
        (min_lat, max_lat)
    lon_range : tuple
        (min_lon, max_lon)
    spacing : float
        Grid spacing in degrees (default 0.25° = ~25-28 km)
    
    Returns:
    --------
    DataFrame with lat, lon columns
    """
    print(f"\nCreating dense synthetic grid:")
    print(f"  Spacing: {spacing}° (~{spacing * 111:.1f} km)")
    
    lats = np.arange(lat_range[0], lat_range[1] + spacing, spacing)
    lons = np.arange(lon_range[0], lon_range[1] + spacing, spacing)
    
    # Create all combinations
    grid_points = [(lat, lon) for lat in lats for lon in lons]
    df_grid = pd.DataFrame(grid_points, columns=['lat', 'lon'])
    
    print(f"  Latitude points: {len(lats)}")
    print(f"  Longitude points: {len(lons)}")
    print(f"  Total grid points: {len(df_grid)}")
    
    return df_grid


def extract_merra2_meteorology(output_dir="nasa_power_data"):
    """Extract MERRA-2 meteorology data at native resolution"""
    
    print("="*80)
    print("EXTRACTING MERRA-2 METEOROLOGY DATA")
    print("="*80)
    
    filepath = 'https://nasa-power.s3.us-west-2.amazonaws.com/merra2/temporal/power_merra2_daily_temporal_lst.zarr'
    
    print("\nConnecting to MERRA-2 Zarr datastore...")
    try:
        filepath_mapped = fsspec.get_mapper(filepath)
        ds = xr.open_zarr(filepath_mapped, consolidated=True)
        print("✓ Successfully connected to MERRA-2 datastore")
    except Exception as e:
        print(f"✗ Error: {e}")
        return None
    
    # Display grid info
    print(f"\nGrid resolution: ~{(ds.lat.values[1] - ds.lat.values[0]):.4f}° lat × {(ds.lon.values[1] - ds.lon.values[0]):.4f}° lon")
    
    # Variables to extract
    vars_to_keep = [
        "T2M", "T2MDEW", "T2MWET", "T2M_MAX", "T2M_MIN", "T2M_RANGE", "TS",
        "RH2M", "QV2M", "PRECTOTCORR",
        "WS2M", "WS10M", "WS50M", "WD2M", "WD10M", "WD50M",
        "PS", "GWETROOT", "GWETTOP", "EVPTRNS"
    ]
    
    available_vars = [var for var in vars_to_keep if var in ds.data_vars]
    print(f"✓ Extracting {len(available_vars)} variables")
    
    # Define spatial extent
    lat_min, lat_max = 45.53, 49.0
    lon_min, lon_max = -124.77, -116.92
    
    # Get grid points
    lat_mask = (ds.lat >= lat_min) & (ds.lat <= lat_max)
    lon_mask = (ds.lon >= lon_min) & (ds.lon <= lon_max)
    
    print(f"\nOriginal grid coverage:")
    print(f"  Lat points: {lat_mask.sum().values}")
    print(f"  Lon points: {lon_mask.sum().values}")
    print(f"  Total locations: {lat_mask.sum().values * lon_mask.sum().values}")
    
    # Extract data
    ds_subset = ds[available_vars]
    time_range = pd.date_range(datetime(2018, 1, 1), datetime(2024, 12, 31), freq='1D')
    
    print(f"\nLoading data... (2-5 minutes)")
    ds_sliced = ds_subset.sel(
        time=time_range,
        lat=ds.lat[lat_mask],
        lon=ds.lon[lon_mask]
    ).load()
    
    print(f"✓ Loaded {ds_sliced.nbytes / 1e6:.2f} MB")
    
    # Convert to DataFrame
    df = ds_sliced.to_dataframe().reset_index()
    
    # Save
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, "merra2_meteorology_native.csv")
    df.to_csv(output_file, index=False)
    
    print(f"✓ Saved: {output_file}")
    print(f"  Rows: {len(df):,}, Locations: {df[['lat', 'lon']].drop_duplicates().shape[0]:,}")
    
    return df


def extract_syn1deg_radiation(output_dir="nasa_power_data"):
    """Extract SYN1deg radiation data at 1° resolution"""
    
    print("\n" + "="*80)
    print("EXTRACTING SYN1DEG RADIATION DATA")
    print("="*80)
    
    filepath = 'https://nasa-power.s3.us-west-2.amazonaws.com/syn1deg/temporal/power_syn1deg_daily_temporal_lst.zarr'
    
    print("\nConnecting to SYN1deg Zarr datastore...")
    filepath_mapped = fsspec.get_mapper(filepath)
    ds = xr.open_zarr(filepath_mapped, consolidated=True)
    print("✓ Successfully connected")
    
    print(f"Grid resolution: ~{(ds.lat.values[1] - ds.lat.values[0]):.4f}° lat × {(ds.lon.values[1] - ds.lon.values[0]):.4f}° lon")
    
    vars_to_keep = [
        "ALLSKY_KT", "ALLSKY_SFC_LW_DWN", "ALLSKY_SFC_LW_UP", "ALLSKY_SFC_PAR_TOT",
        "ALLSKY_SFC_SW_DIFF", "ALLSKY_SFC_SW_DNI", "ALLSKY_SFC_SW_DWN", "ALLSKY_SFC_SW_UP",
        "ALLSKY_SFC_UV_INDEX", "ALLSKY_SFC_UVA", "ALLSKY_SFC_UVB", "ALLSKY_SRF_ALB",
        "AOD_55", "AOD_55_ADJ", "CLOUD_AMT", "CLOUD_AMT_DAY", "CLOUD_AMT_NIGHT",
        "CLOUD_OD", "CLRSKY_DAYS", "CLRSKY_KT", "CLRSKY_SFC_LW_DWN", "CLRSKY_SFC_LW_UP",
        "CLRSKY_SFC_PAR_TOT", "CLRSKY_SFC_SW_DIFF", "CLRSKY_SFC_SW_DWN", "CLRSKY_SFC_SW_UP",
        "CLRSKY_SRF_ALB", "MIDDAY_INSOL", "TOA_SW_DNI", "TOA_SW_DWN"
    ]
    
    lat_min, lat_max = 45.53, 49.0
    lon_min, lon_max = -124.77, -116.92
    
    lat_mask = (ds.lat >= lat_min) & (ds.lat <= lat_max)
    lon_mask = (ds.lon >= lon_min) & (ds.lon <= lon_max)
    
    print(f"\nOriginal grid coverage:")
    print(f"  Lat points: {lat_mask.sum().values}")
    print(f"  Lon points: {lon_mask.sum().values}")
    print(f"  Total locations: {lat_mask.sum().values * lon_mask.sum().values}")
    
    ds_subset = ds[vars_to_keep]
    time_range = pd.date_range(datetime(2018, 1, 1), datetime(2024, 12, 31), freq='1D')
    
    print(f"\nLoading data... (2-5 minutes)")
    ds_sliced = ds_subset.sel(
        time=time_range,
        lat=ds.lat[lat_mask],
        lon=ds.lon[lon_mask]
    ).load()
    
    print(f"✓ Loaded {ds_sliced.nbytes / 1e6:.2f} MB")
    
    df = ds_sliced.to_dataframe().reset_index()
    
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, "syn1deg_radiation_native.csv")
    df.to_csv(output_file, index=False)
    
    print(f"✓ Saved: {output_file}")
    print(f"  Rows: {len(df):,}, Locations: {df[['lat', 'lon']].drop_duplicates().shape[0]:,}")
    
    return df


def interpolate_to_dense_grid(df_source, df_grid, time_col='time', 
                               exclude_cols=['time', 'lat', 'lon']):
    """
    Interpolate source data to dense grid using spatial interpolation
    
    Parameters:
    -----------
    df_source : DataFrame
        Source data with lat, lon, time, and variables
    df_grid : DataFrame
        Target grid with lat, lon columns
    time_col : str
        Name of time column
    exclude_cols : list
        Columns to exclude from interpolation
    
    Returns:
    --------
    DataFrame with interpolated data on dense grid
    """
    print(f"\nInterpolating data to dense grid:")
    print(f"  Source locations: {df_source[['lat', 'lon']].drop_duplicates().shape[0]}")
    print(f"  Target locations: {len(df_grid)}")
    
    # Get all variable columns (exclude time, lat, lon)
    var_cols = [col for col in df_source.columns if col not in exclude_cols]
    print(f"  Variables to interpolate: {len(var_cols)}")
    
    # Get unique time points
    unique_times = df_source[time_col].unique()
    print(f"  Time points: {len(unique_times)}")
    
    result_list = []
    
    # Process in batches to show progress
    print(f"\nInterpolating (this may take 5-15 minutes)...")
    
    # Sample first to estimate time
    if len(unique_times) > 100:
        print("  Processing in batches with progress bar...")
        batch_size = 50
    else:
        batch_size = len(unique_times)
    
    for i in tqdm(range(0, len(unique_times), batch_size), desc="Interpolating"):
        batch_times = unique_times[i:i+batch_size]
        
        for time_val in batch_times:
            # Get source data for this time
            df_time = df_source[df_source[time_col] == time_val].copy()
            
            if len(df_time) == 0:
                continue
            
            # Source coordinates
            source_points = df_time[['lat', 'lon']].values
            
            # Target coordinates
            target_points = df_grid[['lat', 'lon']].values
            
            # Create result dataframe for this time
            df_result = df_grid.copy()
            df_result[time_col] = time_val
            
            # Interpolate each variable
            for var in var_cols:
                source_values = df_time[var].values
                
                # Use linear interpolation (faster) with nearest neighbor fallback
                interpolated = griddata(
                    source_points,
                    source_values,
                    target_points,
                    method='linear',
                    fill_value=np.nan
                )
                
                # Fill remaining NaNs with nearest neighbor
                nan_mask = np.isnan(interpolated)
                if nan_mask.any():
                    nearest = griddata(
                        source_points,
                        source_values,
                        target_points[nan_mask],
                        method='nearest'
                    )
                    interpolated[nan_mask] = nearest
                
                df_result[var] = interpolated
            
            result_list.append(df_result)
    
    # Combine all time steps
    df_interpolated = pd.concat(result_list, ignore_index=True)
    
    print(f"\n✓ Interpolation complete!")
    print(f"  Output rows: {len(df_interpolated):,}")
    print(f"  Output locations: {df_interpolated[['lat', 'lon']].drop_duplicates().shape[0]}")
    
    return df_interpolated


def main():
    """Main execution function"""
    
    print("\n" + "="*80)
    print("HIGH-DENSITY CLIMATE DATA EXTRACTION WITH INTERPOLATION")
    print("="*80)
    print("\nThis script will:")
    print("  1. Extract MERRA-2 meteorology at native resolution (~0.5°)")
    print("  2. Extract SYN1deg radiation at native resolution (1°)")
    print("  3. Create dense synthetic grid (0.25° spacing)")
    print("  4. Interpolate both datasets to dense grid")
    print("  5. Merge into single dataset")
    print("\nExpected output: 100-200+ locations")
    print("Estimated time: 15-30 minutes")
    print("Estimated file size: 200-500 MB")
    print("="*80)
    
    output_dir = "nasa_power_data"
    os.makedirs(output_dir, exist_ok=True)
    
    # Step 1: Extract MERRA-2 meteorology
    print("\n\nSTEP 1/5: Extracting MERRA-2 meteorology...")
    df_met = extract_merra2_meteorology(output_dir)
    
    if df_met is None:
        print("✗ Failed to extract meteorology data. Exiting.")
        return
    
    # Step 2: Extract SYN1deg radiation
    print("\n\nSTEP 2/5: Extracting SYN1deg radiation...")
    df_rad = extract_syn1deg_radiation(output_dir)
    
    # Step 3: Create dense grid
    print("\n\nSTEP 3/5: Creating dense synthetic grid...")
    lat_range = (45.53, 49.0)
    lon_range = (-124.77, -116.92)
    
    # You can adjust spacing here:
    # 0.5° = ~50-55 km (fewer locations, faster)
    # 0.25° = ~25-28 km (more locations, recommended)
    # 0.1° = ~10-11 km (very dense, slower)
    grid_spacing = 0.25
    
    df_dense_grid = create_dense_grid(lat_range, lon_range, spacing=grid_spacing)
    
    # Step 4a: Interpolate meteorology to dense grid
    print("\n\nSTEP 4a/5: Interpolating meteorology to dense grid...")
    df_met_interp = interpolate_to_dense_grid(df_met, df_dense_grid, time_col='time')
    
    # Save intermediate result
    met_interp_file = os.path.join(output_dir, "merra2_meteorology_interpolated.csv")
    print(f"\nSaving interpolated meteorology: {met_interp_file}")
    df_met_interp.to_csv(met_interp_file, index=False)
    print(f"✓ Saved ({os.path.getsize(met_interp_file) / 1e6:.2f} MB)")
    
    # Step 4b: Interpolate radiation to dense grid
    print("\n\nSTEP 4b/5: Interpolating radiation to dense grid...")
    df_rad_interp = interpolate_to_dense_grid(df_rad, df_dense_grid, time_col='time')
    
    # Save intermediate result
    rad_interp_file = os.path.join(output_dir, "syn1deg_radiation_interpolated.csv")
    print(f"\nSaving interpolated radiation: {rad_interp_file}")
    df_rad_interp.to_csv(rad_interp_file, index=False)
    print(f"✓ Saved ({os.path.getsize(rad_interp_file) / 1e6:.2f} MB)")
    
    # Step 5: Merge interpolated datasets
    print("\n\nSTEP 5/5: Merging interpolated datasets...")
    
    # Round coordinates to handle floating point precision
    df_met_interp['lat'] = df_met_interp['lat'].round(4)
    df_met_interp['lon'] = df_met_interp['lon'].round(4)
    df_rad_interp['lat'] = df_rad_interp['lat'].round(4)
    df_rad_interp['lon'] = df_rad_interp['lon'].round(4)
    
    # Merge
    df_merged = pd.merge(
        df_met_interp,
        df_rad_interp,
        on=['time', 'lat', 'lon'],
        how='inner'
    )
    
    print(f"\n✓ Merge complete!")
    print(f"  Combined rows: {len(df_merged):,}")
    print(f"  Combined locations: {df_merged[['lat', 'lon']].drop_duplicates().shape[0]:,}")
    print(f"  Total columns: {len(df_merged.columns)}")
    
    # Save final merged dataset
    output_file = os.path.join(output_dir, "combined_climate_data_dense.csv")
    print(f"\nSaving final combined dataset: {output_file}")
    df_merged.to_csv(output_file, index=False)
    print(f"✓ Saved ({os.path.getsize(output_file) / 1e6:.2f} MB)")
    
    # Final summary
    print("\n\n" + "="*80)
    print("EXTRACTION COMPLETE!")
    print("="*80)
    
    unique_locs = df_merged[['lat', 'lon']].drop_duplicates()
    
    print(f"\n📊 FINAL DATASET SUMMARY:")
    print(f"  • Total data points: {len(df_merged):,}")
    print(f"  • Unique locations: {len(unique_locs):,}")
    print(f"  • Date range: {df_merged['time'].min()} to {df_merged['time'].max()}")
    print(f"  • Total parameters: {len(df_merged.columns) - 3}")
    
    print(f"\n📍 SPATIAL COVERAGE:")
    print(f"  • Latitude range: {unique_locs['lat'].min():.4f}° to {unique_locs['lat'].max():.4f}°N")
    print(f"  • Longitude range: {unique_locs['lon'].min():.4f}° to {unique_locs['lon'].max():.4f}°W")
    print(f"  • Grid spacing: ~{grid_spacing}° (~{grid_spacing * 111:.1f} km)")
    
    print(f"\n📁 OUTPUT FILES:")
    print(f"  1. {os.path.join(output_dir, 'merra2_meteorology_native.csv')}")
    print(f"  2. {os.path.join(output_dir, 'syn1deg_radiation_native.csv')}")
    print(f"  3. {os.path.join(output_dir, 'merra2_meteorology_interpolated.csv')}")
    print(f"  4. {os.path.join(output_dir, 'syn1deg_radiation_interpolated.csv')}")
    print(f"  5. {os.path.join(output_dir, 'combined_climate_data_dense.csv')} ⭐ USE THIS")
    
    print(f"\n✅ Ready for preprocessing and clustering!")
    print(f"   Next: Run your preprocessing pipeline on combined_climate_data_dense.csv")
    print("="*80)


if __name__ == "__main__":
    # Install required packages if not already installed
    try:
        from tqdm.auto import tqdm
    except ImportError:
        print("Installing tqdm for progress bars...")
        os.system("pip install tqdm")
        from tqdm.auto import tqdm
    
    try:
        from scipy.interpolate import griddata
    except ImportError:
        print("Installing scipy for interpolation...")
        os.system("pip install scipy")
        from scipy.interpolate import griddata
    
    main()


HIGH-DENSITY CLIMATE DATA EXTRACTION WITH INTERPOLATION

This script will:
  1. Extract MERRA-2 meteorology at native resolution (~0.5°)
  2. Extract SYN1deg radiation at native resolution (1°)
  3. Create dense synthetic grid (0.25° spacing)
  4. Interpolate both datasets to dense grid
  5. Merge into single dataset

Expected output: 100-200+ locations
Estimated time: 15-30 minutes
Estimated file size: 200-500 MB


STEP 1/5: Extracting MERRA-2 meteorology...
EXTRACTING MERRA-2 METEOROLOGY DATA

Connecting to MERRA-2 Zarr datastore...
✓ Successfully connected to MERRA-2 datastore

Grid resolution: ~0.5000° lat × 0.6250° lon
✓ Extracting 20 variables

Original grid coverage:
  Lat points: 7
  Lon points: 12
  Total locations: 84

Loading data... (2-5 minutes)
✓ Loaded 17.20 MB
✓ Saved: nasa_power_data/merra2_meteorology_native.csv
  Rows: 214,788, Locations: 84


STEP 2/5: Extracting SYN1deg radiation...

EXTRACTING SYN1DEG RADIATION DATA

Connecting to SYN1deg Zarr datastore...
✓ Succ

Interpolating:   0%|          | 0/52 [00:00<?, ?it/s]


✓ Interpolation complete!
  Output rows: 1,265,715
  Output locations: 495

Saving interpolated meteorology: nasa_power_data/merra2_meteorology_interpolated.csv
✓ Saved (478.15 MB)


STEP 4b/5: Interpolating radiation to dense grid...

Interpolating data to dense grid:
  Source locations: 24
  Target locations: 495
  Variables to interpolate: 30
  Time points: 2557

Interpolating (this may take 5-15 minutes)...
  Processing in batches with progress bar...


Interpolating:   0%|          | 0/52 [00:00<?, ?it/s]


✓ Interpolation complete!
  Output rows: 1,265,715
  Output locations: 495

Saving interpolated radiation: nasa_power_data/syn1deg_radiation_interpolated.csv
✓ Saved (676.38 MB)


STEP 5/5: Merging interpolated datasets...

✓ Merge complete!
  Combined rows: 1,265,715
  Combined locations: 495
  Total columns: 53

Saving final combined dataset: nasa_power_data/combined_climate_data_dense.csv
✓ Saved (1122.88 MB)


EXTRACTION COMPLETE!

📊 FINAL DATASET SUMMARY:
  • Total data points: 1,265,715
  • Unique locations: 495
  • Date range: 2018-01-01 00:00:00 to 2024-12-31 00:00:00
  • Total parameters: 50

📍 SPATIAL COVERAGE:
  • Latitude range: 45.5300° to 49.0300°N
  • Longitude range: -124.7700° to -116.7700°W
  • Grid spacing: ~0.25° (~27.8 km)

📁 OUTPUT FILES:
  1. nasa_power_data/merra2_meteorology_native.csv
  2. nasa_power_data/syn1deg_radiation_native.csv
  3. nasa_power_data/merra2_meteorology_interpolated.csv
  4. nasa_power_data/syn1deg_radiation_interpolated.csv
  5. nasa_powe

In [5]:
import pandas as pd

df = pd.read_csv('nasa_power_data/combined_climate_data_dense.csv')

print(f"Total rows: {len(df):,}")
print(f"Unique locations: {df[['lat', 'lon']].drop_duplicates().shape[0]:,}")
print(f"Date range: {df['time'].min()} to {df['time'].max()}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nLocation distribution:")
print(f"  Lat range: {df['lat'].min():.2f} to {df['lat'].max():.2f}")
print(f"  Lon range: {df['lon'].min():.2f} to {df['lon'].max():.2f}")

Total rows: 1,265,715
Unique locations: 495
Date range: 2018-01-01 to 2024-12-31
Missing values: 0

Location distribution:
  Lat range: 45.53 to 49.03
  Lon range: -124.77 to -116.77
